In [ ]:
import numpy as np

from collections import Counter

from datasets import load_from_disk
from datasets.combine import concatenate_datasets

from torchvision.transforms import v2 as torch_augmentation

from tqdm import tqdm

In [ ]:
DATASET_PATH = "/home/sulcm/datasets/milk10k/milk10k"

In [ ]:
lds = load_from_disk(dataset_path=DATASET_PATH)
lds.cleanup_cache_files()

In [ ]:
ds = lds["train"]

In [ ]:
labels_counts = Counter(ds["label"])

most_common_label_id, max_label_count = labels_counts.most_common(1)[0]

In [ ]:
labels_counts

In [ ]:
image_resolutions = set([im.size for im in ds["image"]])

In [ ]:
offline_augmentation = torch_augmentation.Compose([
    torch_augmentation.RandomHorizontalFlip(),
    torch_augmentation.RandomRotation([-360, 360]),
    torch_augmentation.ColorJitter(brightness=0.1, contrast=0.1),
    torch_augmentation.RandomChoice([
        torch_augmentation.RandomResizedCrop(size, scale=(0.8, 1.0)) for size in image_resolutions
    ])
])


def apply_offline_augmentation(batch):
    batch["image"] = [offline_augmentation(im.convert("RGB")) for im in batch["image"]]
    return batch

In [ ]:
ds_per_label = []
for l, c in tqdm(labels_counts.items(), unit="label", desc="Dataset augmentation"):
    _cls_samples = ds.filter(lambda cols: [v == l for v in cols["label"]], batched=True)
    if l != most_common_label_id:
        repeat_count = (max_label_count // c) - 1
        if repeat_count > 0:
            # Repeat class in full lenght
            repeated_samples = _cls_samples.repeat(repeat_count)
        else:
            # Create empty placeholder
            repeated_samples = _cls_samples.repeat(0)

        if max_label_count - (c + len(repeated_samples)) >= max_label_count // 100:
            # Fill-out remaining samples if there is greater then a hundredth difference in lenght between most common class and current one
            _rnd_select = np.random.choice(
                len(_cls_samples),
                ((max_label_count - (c + len(repeated_samples))) // 8) * 8,
                replace=False
            )
            padding_samples = _cls_samples.select(_rnd_select)
            repeated_samples = concatenate_datasets([repeated_samples, padding_samples])

        repeated_samples = repeated_samples.map(apply_offline_augmentation, batched=True, batch_size=512, num_proc=4, desc=f"Augmentation of repeated samples from {l} class")
        print(f"Adding class {l} (with augmentations)")
        ds_per_label.append(
            concatenate_datasets([_cls_samples, repeated_samples])
        )
    else:
        print(f"Adding most common class {l} (without augmentations)")
        ds_per_label.append(_cls_samples)

In [ ]:
augmented_dataset = concatenate_datasets(ds_per_label)

In [ ]:
label_8 = ds.filter(lambda cols: [v == 8 for v in cols["label"]], batched=True)

In [ ]:
prev = label_8[0]

In [ ]:
def apply_offline_augmentation(batch):
    batch["image"] = offline_augmentation(batch["image"])
    return batch

In [ ]:
t_mapped = label_8.map(apply_offline_augmentation, batched=True, batch_size=10)

In [ ]:
aug_ims = [offline_augmentation(im) for im in label_8["image"]]

In [ ]:
aug_ims[9]

In [ ]:
label_8.select([0,1,2])

In [ ]:
((max_label_count - 4000) // 10) * 10

In [ ]:
((max_label_count - len(label_8)) // 10) * 10

In [ ]:
np.random.choice(4000, 1040, replace=False)